# Kruskal-Wallis 检验完整教程：非参数方差分析

## 📚 教学目标
1. 理解 Kruskal-Wallis 检验是单因素 ANOVA 的非参数替代
2. 掌握检验的假设设定和计算步骤
3. 用 scipy.stats.kruskal 进行检验
4. 理解事后比较方法
5. 对比 Kruskal-Wallis 与 ANOVA 的结果

**参考书**：《基础统计学(第14版)》（Triola）第 13-5 节
**教学策略**：先回顾 ANOVA 的局限性，再引入 Kruskal-Wallis 作为非参数替代

---

## 1. 场景设定：三种肥料的效果比较

### 🎯 问题

农业实验比较三种肥料对作物产量的影响：
- 肥料 A: 8 块田地
- 肥料 B: 8 块田地
- 肥料 C: 8 块田地

问题：三种肥料的产量是否相同？

### 📖 书中的定义

> Kruskal-Wallis 检验是单因素 ANOVA 的非参数替代方法。
> 它不依赖于总体分布的假设，适用于数据不满足正态性的情况。

### 📐 Kruskal-Wallis 统计量

$$H = \frac{12}{n(n+1)} \sum \frac{R_i^2}{n_i} - 3(n+1)$$

其中：
- n = 总样本量
- $n_i$ = 第 i 组的样本量
- $R_i$ = 第 i 组的秩和

### 💡 适用条件

| 条件 | 说明 |
|------|------|
| 独立性 | 各组数据相互独立 |
| 随机性 | 数据来自随机抽样 |
| 分布形状 | 各组分布形状相似 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 生成数据与探索

### 📐 数据生成过程 (DGP)

我们模拟一个非正态分布的数据集：
- 肥料 A: 偏态分布 (右偏)
- 肥料 B: 偏态分布 (右偏)
- 肥料 C: 偏态分布 (右偏)

In [ ]:
# ========== 步骤 1: 生成数据 ==========

print("=" * 60)
print("📋 Kruskal-Wallis 检验: 肥料效果比较")
print("=" * 60)

# --- 数据生成过程 (DGP) ---
# 使用偏态分布 (指数分布) 来模拟非正态数据
n_per_group = 8
fertilizer_a = np.random.exponential(20, n_per_group) + 30  # 均值约 50
fertilizer_b = np.random.exponential(25, n_per_group) + 35  # 均值约 60
fertilizer_c = np.random.exponential(20, n_per_group) + 40  # 均值约 60

print(f"\n📊 数据概况:")
print(f"{'肥料':<6} {'n':<6} {'均值':<10} {'中位数':<10} {'标准差':<10}")
print("-" * 42)
for name, data in [('A', fertilizer_a), ('B', fertilizer_b), ('C', fertilizer_c)]:
    print(f"  {name:<6} {len(data):<6} {np.mean(data):<10.2f} {np.median(data):<10.2f} {np.std(data, ddof=1):<10.2f}")

# 正态性检验
print(f"\n📊 正态性检验 (Shapiro-Wilk):")
for name, data in [('A', fertilizer_a), ('B', fertilizer_b), ('C', fertilizer_c)]:
    w_stat, p_val = stats.shapiro(data)
    result = '正态' if p_val > 0.05 else '非正态'
    print(f"  肥料 {name}: W = {w_stat:.4f}, p = {p_val:.4f} ({result})")

print(f"\n💡 数据不满足正态性，应使用 Kruskal-Wallis 检验")

print("\n" + "=" * 60)

In [ ]:
# ========== 可视化: 数据分布 ==========

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 箱形图
ax1 = axes[0]
bp = ax1.boxplot([fertilizer_a, fertilizer_b, fertilizer_c], 
                labels=['Fertilizer A', 'Fertilizer B', 'Fertilizer C'],
                patch_artist=True)
colors = ['#2ecc71', '#3498db', '#e74c3c']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax1.set_ylabel('Yield', fontsize=12)
ax1.set_title('Yield by Fertilizer Type', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# 直方图
ax2 = axes[1]
for name, data, color in zip(['A', 'B', 'C'], [fertilizer_a, fertilizer_b, fertilizer_c], colors):
    ax2.hist(data, bins=6, alpha=0.5, color=color, label=f'Fertilizer {name}', density=True, edgecolor='white')
ax2.set_xlabel('Yield', fontsize=12)
ax2.set_ylabel('Density', fontsize=12)
ax2.set_title('Distribution of Yields', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 图解说明：")
print("  左图：箱形图显示各组的中位数和四分位数")
print("  右图：直方图显示数据的分布形态")
print("  数据呈右偏分布，不满足正态性假设")

---

## 3. 手算 Kruskal-Wallis 检验

### 📐 计算步骤

1. 合并所有数据
2. 按大小排序，分配秩次
3. 计算各组的秩和
4. 计算 H 统计量
5. 查表或计算 p 值

In [ ]:
# ========== 步骤 2: 手算 Kruskal-Wallis ==========

print("=" * 60)
print("📋 手算 Kruskal-Wallis 检验")
print("=" * 60)

# 合并数据
all_data = np.concatenate([fertilizer_a, fertilizer_b, fertilizer_c])
group_labels = ['A'] * n_per_group + ['B'] * n_per_group + ['C'] * n_per_group
n_total = len(all_data)
k = 3  # 组数

# 排序并分配秩次
sorted_indices = np.argsort(all_data)
sorted_data = all_data[sorted_indices]
sorted_labels = [group_labels[i] for i in sorted_indices]

# 处理并列值
ranks = np.zeros(n_total)
i = 0
while i < n_total:
    j = i
    while j < n_total and sorted_data[j] == sorted_data[i]:
        j += 1
    avg_rank = (i + j + 1) / 2
    for m in range(i, j):
        ranks[sorted_indices[m]] = avg_rank
    i = j

print(f"\n📊 步骤 1: 合并排序并分配秩次")
print(f"  总样本量 n = {n_total}")
print(f"  组数 k = {k}")

# 计算各组秩和
rank_sums = {}
for i, name in enumerate(['A', 'B', 'C']):
    start_idx = i * n_per_group
    end_idx = start_idx + n_per_group
    rank_sums[name] = np.sum(ranks[start_idx:end_idx])

print(f"\n📊 步骤 2: 计算各组秩和")
for name in ['A', 'B', 'C']:
    print(f"  肥料 {name}: R = {rank_sums[name]:.1f}")

# 计算 H 统计量
H_manual = (12 / (n_total * (n_total + 1))) * sum(rank_sums[name]**2 / n_per_group for name in ['A', 'B', 'C']) - 3 * (n_total + 1)

print(f"\n📊 步骤 3: 计算 H 统计量")
print(f"  H = (12 / (n(n+1))) × Σ(R²/nᵢ) - 3(n+1)")
print(f"  H = {H_manual:.4f}")

# 自由度
df = k - 1
print(f"\n📊 步骤 4: 自由度")
print(f"  df = k - 1 = {df}")

# p 值
p_value = 1 - stats.chi2.cdf(H_manual, df)

print(f"\n📊 步骤 5: p 值")
print(f"  p = {p_value:.6f}")

# 判断
alpha = 0.05
if p_value < alpha:
    print(f"\n  💡 p < {alpha}，拒绝 H₀：至少有一种肥料的效果不同")
else:
    print(f"\n  💡 p ≥ {alpha}，不能拒绝 H₀")

print("\n" + "=" * 60)

In [ ]:
# ========== 步骤 3: scipy 验证 ==========

print("=" * 60)
print("🔬 scipy.stats.kruskal 验证")
print("=" * 60)

h_scipy, p_scipy = stats.kruskal(fertilizer_a, fertilizer_b, fertilizer_c)

print(f"\n📊 手算 vs scipy 对比:")
print(f"  手算 H = {H_manual:.6f}")
print(f"  scipy H = {h_scipy:.6f}")
print(f"  手算 p = {p_value:.6f}")
print(f"  scipy p = {p_scipy:.6f}")
print(f"\n  ✅ 验证通过！")

print("\n" + "=" * 60)

---

## 4. 事后比较

### 📖 书中的要点

> 当 Kruskal-Wallis 检验拒绝 H₀ 时，需要进行事后比较来确定
> 具体哪些组之间存在差异。

### 📐 Dunn 检验

Dunn 检验是 Kruskal-Wallis 事后比较的常用方法，需要对多重比较进行校正。

In [ ]:
# ========== 步骤 4: 事后比较 ==========

print("=" * 60)
print("📋 事后比较: 两两秩和检验")
print("=" * 60)

# 两两比较 (Wilcoxon 秩和检验)
pairs = [('A', 'B', fertilizer_a, fertilizer_b),
         ('A', 'C', fertilizer_a, fertilizer_c),
         ('B', 'C', fertilizer_b, fertilizer_c)]

alpha = 0.05
m = 3  # 比较次数
alpha_bonferroni = alpha / m

print(f"\n📊 Bonferroni 校正:")
print(f"  原始 α = {alpha}")
print(f"  比较次数 m = {m}")
print(f"  校正后 α = {alpha_bonferroni:.4f}")

print(f"\n📊 两两秩和检验:")
print(f"{'比较':<15} {'z 统计量':<12} {'p 值':<12} {'校正后 p':<12} {'结论':<10}")
print("-" * 61)

for name1, name2, data1, data2 in pairs:
    z_stat, p_val = stats.ranksums(data1, data2)
    p_corrected = min(p_val * m, 1.0)  # Bonferroni 校正
    sig = '显著' if p_corrected < alpha else '不显著'
    print(f"  {name1} vs {name2}     {z_stat:<12.4f} {p_val:<12.4f} {p_corrected:<12.4f} {sig}")

print(f"\n💡 解读:")
print(f"  Bonferroni 校正后，只有 p < {alpha_bonferroni:.4f} 才显著")
print(f"  校正控制了多重比较的总体错误率")

print("\n" + "=" * 60)

---

## 5. ANOVA vs Kruskal-Wallis 对比

### 💡 核心对比

我们同时进行 ANOVA 和 Kruskal-Wallis 检验，比较结果。

In [ ]:
# ========== 步骤 5: ANOVA vs Kruskal-Wallis ==========

print("=" * 60)
print("📋 ANOVA vs Kruskal-Wallis 对比")
print("=" * 60)

# ANOVA
f_stat, f_p = stats.f_oneway(fertilizer_a, fertilizer_b, fertilizer_c)

# Kruskal-Wallis
h_stat, h_p = stats.kruskal(fertilizer_a, fertilizer_b, fertilizer_c)

print(f"\n📊 检验结果对比:")
print(f"{'检验方法':<20} {'统计量':<15} {'p 值':<15}")
print("-" * 50)
print(f"  {'ANOVA':<18} {'F = ' + f'{f_stat:.4f}':<15} {f_p:<15.6f}")
print(f"  {'Kruskal-Wallis':<18} {'H = ' + f'{h_stat:.4f}':<15} {h_p:<15.6f}")

alpha = 0.05
print(f"\n📊 判断 (α = {alpha}):")
print(f"  ANOVA: {'拒绝 H₀' if f_p < alpha else '不能拒绝 H₀'}")
print(f"  Kruskal-Wallis: {'拒绝 H₀' if h_p < alpha else '不能拒绝 H₀'}")

print(f"\n💡 解读:")
print(f"  当数据满足正态性时，ANOVA 的功效更高")
print(f"  当数据不满足正态性时，Kruskal-Wallis 更稳健")
print(f"  本例中数据非正态，应优先使用 Kruskal-Wallis")

print("\n" + "=" * 60)

In [ ]:
# ========== 可视化: 对比 ==========

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 箱形图
ax1 = axes[0]
bp = ax1.boxplot([fertilizer_a, fertilizer_b, fertilizer_c], 
                labels=['Fertilizer A', 'Fertilizer B', 'Fertilizer C'],
                patch_artist=True)
colors = ['#2ecc71', '#3498db', '#e74c3c']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax1.set_ylabel('Yield', fontsize=12)
ax1.set_title('Yield by Fertilizer Type', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# 秩次分布
ax2 = axes[1]
rank_sums_list = [rank_sums['A'], rank_sums['B'], rank_sums['C']]
bars = ax2.bar(['Fertilizer A', 'Fertilizer B', 'Fertilizer C'], rank_sums_list, 
               color=colors, alpha=0.7, edgecolor='white')
ax2.set_ylabel('Rank Sum', fontsize=12)
ax2.set_title('Rank Sums by Fertilizer Type', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# 标注数值
for bar, val in zip(bars, rank_sums_list):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
            f'{val:.0f}', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

print("💡 图解说明：")
print("  左图：各组产量的箱形图")
print("  右图：各组的秩和")
print(f"  Kruskal-Wallis H = {h_stat:.4f}, p = {h_p:.4f}")
print("  秩和差异越大，越可能拒绝 H₀")

---

## 6. 模拟验证

### 💡 核心思想

如果 H₀ 为真（三组分布相同），多次重复实验得到的 H 统计量应该服从卡方分布。

In [ ]:
# ========== 步骤 6: 模拟验证 ==========

print("=" * 60)
print("📋 模拟验证: H 统计量的抽样分布")
print("=" * 60)

# 模拟参数
n_sims = 10000
n_sim = 8  # 每组样本量

# 存储每次模拟的 H 值
h_simulated = []

for _ in range(n_sims):
    # 生成来自同一分布的数据 (H₀ 为真)
    group1 = np.random.exponential(20, n_sim) + 30
    group2 = np.random.exponential(20, n_sim) + 30
    group3 = np.random.exponential(20, n_sim) + 30
    
    # 计算 Kruskal-Wallis
    h_val, _ = stats.kruskal(group1, group2, group3)
    h_simulated.append(h_val)

h_simulated = np.array(h_simulated)

# 可视化
fig, ax = plt.subplots(figsize=(10, 6))

# 模拟分布
ax.hist(h_simulated, bins=50, density=True, alpha=0.6, color='steelblue', edgecolor='white', label='Simulated H')

# 理论卡方分布
x_chi2 = np.linspace(0, 15, 1000)
y_chi2 = stats.chi2.pdf(x_chi2, df=2)
ax.plot(x_chi2, y_chi2, 'r-', linewidth=2, label='Theoretical χ² (df=2)')

# 标记我们的观测值
ax.axvline(x=h_stat, color='#e67e22', linestyle='--', linewidth=2, label=f'Our H = {h_stat:.2f}')

ax.set_xlabel('H Value', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Simulated vs Theoretical Distribution of H', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"💡 图解说明：")
print(f"  蓝色直方图：{n_sims} 次模拟的 H 统计量分布")
print(f"  红色曲线：理论卡方分布 (df=2)")
print(f"  橙色虚线：我们观测到的 H = {h_stat:.2f}")
print(f"  模拟分布与理论分布高度吻合，验证了检验的理论基础")

---

## 📌 核心概念回顾

### 📌 Kruskal-Wallis 检验
- **定义**: 单因素 ANOVA 的非参数替代方法
- **公式**: $H = \frac{12}{n(n+1)} \sum \frac{R_i^2}{n_i} - 3(n+1)$
- **含义**: H 值越大，各组之间的秩次差异越大
- **判断标准**: p < α 时拒绝 H₀

### 📌 与 ANOVA 的对比
- **ANOVA**: 检验均值差异，需要正态性
- **Kruskal-Wallis**: 检验分布位置差异，不需要正态性
- **选择依据**: 数据是否满足正态性假设

### 📌 自由度
- **定义**: df = k - 1
- **含义**: 决定卡方分布的形态

### 📌 事后比较
- **目的**: Kruskal-Wallis 拒绝 H₀ 后，确定具体哪些组不同
- **方法**: Dunn 检验、两两秩和检验
- **注意**: 多重比较需要校正以控制错误率

### 🔗 完整流程
```
设定假设 → 检查条件 → 计算 H → 求 p 值 → 判断 → 事后比较
    ↓          ↓         ↓        ↓       ↓       ↓
  H₀/H₁   独立/形状   公式    查表/软件  p<α?   Dunn/秩和
```

---

## ❌ 常见误区

### ❌ 误区 1: 在不满足参数检验条件时误用参数方法
**✓ 正确做法**: 当数据严重偏离正态分布时，应使用 Kruskal-Wallis 检验而不是 ANOVA。强行使用 ANOVA 可能导致错误的结论。应先检验正态性（如 Shapiro-Wilk），再选择合适的检验方法。

### ❌ 误区 2: Kruskal-Wallis 检验没有假设
**✓ 正确理解**: Kruskal-Wallis 检验也有假设：独立性、随机抽样、各组分布形状相似。如果分布形状不同，检验可能不适用。

### ❌ 误区 3: Kruskal-Wallis 检验的 p 值与 ANOVA 的 p 值应该相同
**✓ 正确理解**: 两种检验的 p 值通常不同，因为它们检验的假设不同。ANOVA 检验均值差异，Kruskal-Wallis 检验分布位置差异。当数据正态时，两者通常给出一致的结论。

### ❌ 误区 4: 非参数检验可以完全替代参数检验
**✓ 正确理解**: 非参数检验的功效（检测真实差异的能力）通常低于参数检验。当数据满足正态性时，使用非参数检验会浪费信息，降低检验功效。应根据数据特征选择合适的检验方法。

### ❌ 误区 5: 忽略事后比较的多重校正
**✓ 正确做法**: 当 Kruskal-Wallis 拒绝 H₀ 后，需要进行事后比较。但多次比较会增加第一类错误的概率，必须进行校正（如 Bonferroni 校正）。不校正可能导致假阳性结论。